# 279. 完全平方数：BFS、DFS 与 DP

共同状态是剩余值 cur；每步减去一个正平方数，目标是到达 0。12 的最优拆分是 4 + 4 + 4，不能贪心地先取 9。

[原文](https://leetcode.cn/problems/perfect-squares/solution/cong-num-squarechai-jie-bfsdfshe-dp-by-4-ykk0/) · 整理于 2026-10-17。原代码保持原样；补充注释另放单元格。

## BFS：按使用个数分层

同一个剩余值首次入队时，步数已最少；`visited` 记录是否到过。时间 O(n√n)，空间 O(n)。

In [ ]:
# 队列保存剩余值，steps 表示已减去几个平方数。
# 固定 size 后再扩展，避免把下一层混进当前层。
# isqrt(cur) 给出本轮合法平方数的最大平方根。

In [ ]:
from collections import deque
from math import isqrt


class Solution:
    def numSquares(self, n: int) -> int:
        queue = deque([n])
        visited = {n}
        steps = 0

        while queue:
            size = len(queue)

            for _ in range(size):
                cur = queue.popleft()
                if cur == 0:
                    return steps

                for r in range(isqrt(cur), 0, -1):
                    nxt = cur - r * r
                    if nxt in visited:
                        continue

                    visited.add(nxt)
                    queue.append(nxt)

            steps += 1

        return -1

In [ ]:
assert Solution().numSquares(12) == 3
assert Solution().numSquares(13) == 2

## 朴素 DFS：定义返回值

`dfs(cur)` 返回凑出 cur 最少还需几个数；每次选择后加 1。存在大量重复子问题，可能指数增长，仅适合小输入；调用栈 O(n)。

In [ ]:
# cur == 0 时不再需要平方数，返回 0。
# 从大到小只是枚举顺序，每个分支仍参与 min。
# 只传整数，没有共享路径列表，因此不需要撤销选择。

In [ ]:
from math import isqrt


class Solution:
    def numSquares(self, n: int) -> int:
        def dfs(cur):
            if cur == 0:
                return 0

            best = float("inf")
            for r in range(isqrt(cur), 0, -1):
                best = min(best, dfs(cur - r * r) + 1)

            return best

        return dfs(n)

In [ ]:
assert Solution().numSquares(12) == 3
assert Solution().numSquares(13) == 2

## 记忆化 DFS：保存子问题答案

`memo / cache` 保存具体最优值，不能用“访问过就跳过”代替。时间 O(n√n)，缓存与栈空间 O(n)；缓存不消除递归深度风险。

In [ ]:
# 缓存键只有 cur，因为未来可选平方数只由剩余值决定。
# 内层函数每次调用重新定义，不同 n 的缓存相互独立。

In [ ]:
from functools import lru_cache
from math import isqrt


class Solution:
    def numSquares(self, n: int) -> int:
        @lru_cache(maxsize=None)
        def dfs(cur):
            if cur == 0:
                return 0

            best = float("inf")
            for r in range(isqrt(cur), 0, -1):
                best = min(best, dfs(cur - r * r) + 1)

            return best

        return dfs(n)

In [ ]:
assert Solution().numSquares(12) == 3
assert Solution().numSquares(13) == 2

## 迭代 DP：按依赖顺序填表

`dp[i]` 是凑出 i 的最少个数；`dp[0]=0`，其余先设无穷大。依赖下标严格更小，所以 i 升序。时间 O(n√n)，空间 O(n)。

In [ ]:
# 转移：dp[i] = min(dp[i], dp[i-r*r] + 1)。
# 平方数可以重复使用，这是完全背包式的状态含义。
# 不要枚举 0，否则会出现无效的自我依赖。

In [ ]:
from math import isqrt


class Solution:
    def numSquares(self, n: int) -> int:
        dp = [float("inf")] * (n + 1)
        dp[0] = 0

        for i in range(1, n + 1):
            for r in range(1, isqrt(i) + 1):
                dp[i] = min(dp[i], dp[i - r * r] + 1)

        return dp[n]

In [ ]:
assert Solution().numSquares(1) == 1
assert Solution().numSquares(12) == 3
assert Solution().numSquares(13) == 2

## 原文通用模板

以下是需填写辅助函数的模板，保留为阅读片段。`is_goal`、`next_states`、`previous_states` 和 `target` 需按题目定义。

### BFS 模板

```python
from collections import deque


def bfs(start):
    queue = deque([start])
    visited = {start}
    steps = 0

    while queue:
        size = len(queue)  # 固定当前层的数量

        for _ in range(size):
            state = queue.popleft()
            if is_goal(state):
                return steps

            for nxt in next_states(state):
                if nxt in visited:
                    continue

                visited.add(nxt)  # 入队时标记
                queue.append(nxt)

        steps += 1

    return -1  # 队列耗尽，仍未到达目标
```

### DFS 模板

```python
def dfs(state):
    if is_goal(state):
        return 0

    best = float("inf")
    for nxt in next_states(state):
        best = min(best, dfs(nxt) + 1)

    return best
```

### DP 模板

```python
dp = [float("inf")] * (target + 1)
dp[0] = 0

for i in range(1, target + 1):
    for prev in previous_states(i):
        dp[i] = min(dp[i], dp[prev] + 1)

answer = dp[target]
```

迁移时先写清：状态是什么、合法下一步是什么、何时结束、函数或数组格子返回什么。BFS 解决等代价最少步数；记忆化 DFS 与填表 DP 复用同一批子问题答案。